# 📖 Notebook 2: News Feed Generation

When you open Instagram, your feed loads in under 500ms — even though you might follow 1,000 people  
who collectively posted hundreds of times today. How?

The answer is **pre-computed feeds**. Instead of assembling your feed when you open the app,  
Instagram builds it in advance every time someone you follow posts.

## Learning Objectives

By the end of this notebook, you'll understand:
- **Fan-out on Read** — assembling the feed at read time (simple but slow)
- **Fan-out on Write** — pre-computing feeds when posts are created (fast reads)
- **The Celebrity Problem** — why fan-out on write breaks for users with millions of followers
- **Hybrid Approach** — how Instagram combines both strategies in production
- How **Redis sorted sets** store precomputed feeds

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/instagram
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json
import statistics

# ── Connections ───────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ PostgreSQL — {cur.fetchone()[0]} users")
    cur.execute("SELECT COUNT(*) FROM posts")
    print(f"   {cur.fetchone()[0]} posts")
    cur.execute("SELECT COUNT(*) FROM follows")
    print(f"   {cur.fetchone()[0]} follow relationships")
    cur.execute("SELECT COUNT(*) FROM precomputed_feed")
    print(f"   {cur.fetchone()[0]} precomputed feed entries")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Core Problem

When User 1 opens their feed, we need to show them recent posts from everyone they follow.

```
User 1 follows: user2, user7, user15, celeb_alice, celeb_bob

We need to:
  1. Find all people User 1 follows
  2. Get recent posts from each of them
  3. Merge and sort by time
  4. Return the top 20

Now imagine 500 MILLION users doing this simultaneously.
```

There are two fundamentally different approaches.

## Strategy 1: Fan-Out on Read ("Pull Model")

**When the user opens their feed**, we:
1. Look up everyone they follow
2. Fetch recent posts from each followed user
3. Merge and sort by timestamp
4. Return the top N

```
User opens feed
      │
      ▼
┌─────────────┐     ┌──────────────┐
│ Get followed │────►│ For EACH     │──► Merge + Sort ──► Return top 20
│ user IDs     │     │ followed user│
│ (1 query)    │     │ get posts    │
│              │     │ (N queries)  │
└─────────────┘     └──────────────┘
```

Let's implement this and see how it performs.

In [ ]:
def get_feed_fan_out_on_read(user_id: int, limit: int = 20) -> list:
    """
    Fan-Out on Read: Build the feed at read time.

    For each user we follow, query their recent posts,
    then merge everything together sorted by time.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # Step 1: Get all users this person follows
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s",
        (user_id,)
    )
    followees = [row["followee_id"] for row in cur.fetchall()]
    step1_time = time.time() - start

    if not followees:
        conn.close()
        return []

    # Step 2: Get recent posts from EACH followed user (one query per user)
    # `media_upload_status = 'complete'` matters: Notebook 1 creates the post
    # row *before* the client uploads the photo, so a feed that skips this
    # filter will happily serve posts whose image does not exist yet.
    all_posts = []
    for followee_id in followees:
        cur.execute(
            """SELECT id, author_id, caption, like_count, created_at
               FROM posts
               WHERE author_id = %s
                 AND media_upload_status = 'complete'
               ORDER BY created_at DESC
               LIMIT 10""",
            (followee_id,)
        )
        all_posts.extend(cur.fetchall())
    step2_time = time.time() - start - step1_time

    # Step 3: Sort by time (newest first) and take top N
    all_posts.sort(key=lambda p: p["created_at"], reverse=True)
    feed = all_posts[:limit]
    total_time = time.time() - start

    conn.close()

    # Stash the cost so we can compare it with fan-out on write further down.
    READ_PATH_STATS.update(queries=1 + len(followees), posts_scanned=len(all_posts))

    print(f"📊 Fan-out on READ stats:")
    print(f"   Following: {len(followees)} users")
    print(f"   Queries: 1 (follows) + {len(followees)} (posts) = {1 + len(followees)} total")
    print(f"   Posts fetched: {len(all_posts)}")
    print(f"   Time: {total_time*1000:.1f}ms (follows: {step1_time*1000:.1f}ms, posts: {step2_time*1000:.1f}ms)")

    return feed

READ_PATH_STATS = {}

# Notebook 1 leaves a post in 'pending' until the client confirms its upload.
# Seed one here (newest post in User 1's network) so we can prove it stays out.
conn = get_db()
cur = conn.cursor()
cur.execute(
    """INSERT INTO posts (author_id, caption, media_type, media_key,
                          media_upload_status, created_at)
       VALUES (11, 'Half-uploaded post — must not be served', 'photo',
               'photos/user_11/pending.jpg', 'pending', NOW())
       RETURNING id"""
)
pending_post_id = cur.fetchone()[0]
conn.commit()
conn.close()

# Let's get User 1's feed using fan-out on read
print("Loading User 1's feed (fan-out on read)...\n")
feed = get_feed_fan_out_on_read(user_id=1, limit=10)
print(f"\nTop 10 posts in feed:")
for i, post in enumerate(feed, 1):
    print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}...")

# ── Invariants the rest of the notebook compares against ────────────────
feed_ids = [p["id"] for p in feed]
assert pending_post_id not in feed_ids, (
    f"post #{pending_post_id} has not finished uploading and must never be in a feed")
times = [p["created_at"] for p in feed]
assert times == sorted(times, reverse=True), "the feed must come back newest-first"
assert READ_PATH_STATS["queries"] > 1, "fan-out on read is the N+1 approach — it must be N+1"
print(f"\n✅ Pending post #{pending_post_id} correctly excluded; "
      f"{READ_PATH_STATS['queries']} DB queries for one feed load")

### Why Fan-Out on Read Doesn't Scale

With our 53 users, it's fast. But imagine Instagram's real scale:

| Metric | Our Demo | Real Instagram |
|--------|----------|---------------|
| Users | 53 | 500,000,000 |
| Avg follows | ~10 | ~500 |
| Feed requests/sec | 1 | 150,000+ |
| DB queries per feed | ~11 | ~501 |

At 150,000 feed requests/second × 501 queries each = **75 million queries/second**.  
That's absurd. No database can handle that.

**The core issue**: we're doing all the expensive work at read time — exactly when users expect instant results.

## Strategy 2: Fan-Out on Write ("Push Model")

Instead of building the feed when the user reads it, we build it when someone **posts**.

When User 7 creates a new post:
1. Save the post to the database (as before)
2. Look up everyone who follows User 7
3. Push this post into each follower's precomputed feed

```
User 7 posts
      │
      ▼
┌─────────────┐     ┌──────────────────┐
│ Save post   │────►│ Get followers    │
│ to database │     │ of User 7        │
└─────────────┘     └────────┬─────────┘
                             │
                    ┌────────▼─────────┐
                    │ Push post into   │
                    │ EACH follower's  │
                    │ precomputed feed │
                    └──────────────────┘
                        │    │    │
                        ▼    ▼    ▼
                    feed:1 feed:3 feed:15  (Redis sorted sets)
```

Now when the user opens their feed, we just read their precomputed feed — **one operation**.

In [ ]:
def fan_out_on_write(author_id: int, post_id: int, created_at_ts: float):
    """
    Fan-Out on Write: When a post is created, push it to all followers' feeds.

    This runs ASYNCHRONOUSLY after the post is saved — the user doesn't wait.
    We use Redis sorted sets (ZSET) where:
    - Key = feed:{user_id}
    - Member = post_id
    - Score = timestamp (for chronological ordering)
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()
    start = time.time()

    # Get all followers of this author
    cur.execute(
        "SELECT follower_id FROM follows WHERE followee_id = %s",
        (author_id,)
    )
    followers = [row[0] for row in cur.fetchall()]

    # Push post_id into each follower's Redis sorted set
    pipeline = r.pipeline()  # batch Redis commands for efficiency
    for follower_id in followers:
        feed_key = f"feed:{follower_id}"
        pipeline.zadd(feed_key, {str(post_id): created_at_ts})
        # Keep only the most recent 500 posts per feed
        pipeline.zremrangebyrank(feed_key, 0, -501)
    pipeline.execute()

    elapsed = (time.time() - start) * 1000
    conn.close()

    print(f"📤 Fan-out on WRITE:")
    print(f"   Post #{post_id} by user {author_id}")
    print(f"   Pushed to {len(followers)} followers' feeds")
    print(f"   Time: {elapsed:.1f}ms")
    return followers

# Simulate User 5 creating a new post
conn = get_db()
cur = conn.cursor()
cur.execute(
    """INSERT INTO posts (author_id, caption, media_type, media_key, created_at)
       VALUES (5, 'Fan-out demo post! 🚀', 'photo', 'photos/user_5/fanout_demo.jpg', NOW())
       RETURNING id, extract(epoch from created_at)"""
)
new_post_id, new_post_ts = cur.fetchone()
new_post_ts = float(new_post_ts)  # psycopg2 returns Decimal; Redis needs float
conn.commit()
conn.close()

print(f"Created post #{new_post_id}\n")

# Now fan it out!
followers = fan_out_on_write(author_id=5, post_id=new_post_id, created_at_ts=new_post_ts)

# Fan-out on write is only correct if EVERY follower got the post. A partial
# fan-out is the failure mode that shows up as "my friend's post never appeared".
r = get_redis()
missing = [f for f in followers
           if r.zscore(f"feed:{f}", str(new_post_id)) is None]
assert not missing, f"fan-out missed {len(missing)} followers: {missing[:5]}"
assert followers, "user 5 should have followers in the seed data"
print(f"✅ Verified: post #{new_post_id} is in all {len(followers)} follower feeds")

### Reading the Precomputed Feed

Now reading the feed is **dead simple** — just read from Redis.
One operation, sub-millisecond.

In [ ]:
WRITE_PATH_STATS = {}

def get_feed_fan_out_on_write(user_id: int, limit: int = 20) -> list:
    """
    Read the precomputed feed from Redis.
    Then "hydrate" each post_id with metadata from PostgreSQL.
    """
    r = get_redis()
    start = time.time()

    # Get top N post IDs from Redis (sorted by score = timestamp, descending)
    feed_key = f"feed:{user_id}"
    post_ids = r.zrevrange(feed_key, 0, limit - 1)
    redis_time = (time.time() - start) * 1000

    if not post_ids:
        print(f"Feed for user {user_id} is empty in Redis (not yet populated)")
        return []

    # Hydrate: fetch full post data from PostgreSQL.
    # Same 'complete' filter as the read path — the sorted set is a superset,
    # hydration is where we drop anything that should not be served.
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(
        """SELECT id, author_id, caption, like_count, created_at
           FROM posts
           WHERE id = ANY(%s)
             AND media_upload_status = 'complete'
           ORDER BY created_at DESC""",
        ([int(pid) for pid in post_ids],)
    )
    posts = cur.fetchall()
    total_time = (time.time() - start) * 1000
    conn.close()

    WRITE_PATH_STATS.update(queries=1, redis_ops=1, posts_scanned=len(post_ids))

    print(f"📊 Fan-out on WRITE read stats:")
    print(f"   Redis lookup: {redis_time:.1f}ms ({len(post_ids)} post IDs)")
    print(f"   Hydration query: 1 (batch fetch by IDs)")
    print(f"   Total: {total_time:.1f}ms")

    return posts

# Read the feed for a follower of User 5
# First, find someone who follows User 5
if followers:
    test_user = followers[0]
    print(f"Reading feed for user {test_user} (follows user 5)...\n")
    feed = get_feed_fan_out_on_write(user_id=test_user, limit=10)
    if feed:
        print(f"\nTop posts in feed:")
        for i, post in enumerate(feed, 1):
            print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}")

    # The point of the push model is that read cost stops scaling with how many
    # people you follow. Compare DB round-trips, not wall-clock: at 53 users both
    # paths finish in single-digit milliseconds and timing would just be noise.
    assert WRITE_PATH_STATS["queries"] == 1, WRITE_PATH_STATS
    assert WRITE_PATH_STATS["queries"] < READ_PATH_STATS["queries"], (
        f"push model should need fewer DB queries than pull: "
        f"{WRITE_PATH_STATS['queries']} vs {READ_PATH_STATS['queries']}")
    print(f"\n✅ DB queries per feed load: "
          f"{READ_PATH_STATS['queries']} (pull) → {WRITE_PATH_STATS['queries']} (push)")

### Comparing the Two Approaches

| Metric | Fan-out on Read | Fan-out on Write |
|--------|----------------|------------------|
| **Read speed** | Slow (N+1 queries) | Fast (1 Redis read + 1 batch query) |
| **Write speed** | Fast (just save post) | Slower (push to all followers) |
| **Storage** | No extra storage | Stores feed per user in Redis |
| **Consistency** | Always fresh | May be slightly stale |
| **Best for** | Few followers | Most users |

## 🌟 The Celebrity Problem

Fan-out on write works great for User 5 with 10 followers.  
But what about **celeb_alice** with **1.5 million followers**?

When she posts, we'd need to push that post into **1.5 million Redis sorted sets**.  
That's called **write amplification** — and it's a huge problem.

```
celeb_alice posts a photo
       │
       ▼
Push to 1,500,000 feeds!  ← This takes seconds or minutes
       │                     and hammers Redis with writes
       ▼
feed:1, feed:2, feed:3, ... feed:1500000
```

The numbers at Instagram's scale:
- Cristiano Ronaldo: 600M+ followers
- One post → 600 million Redis writes
- That's **insane**

In [ ]:
# The celebrity problem is usually asserted ("that's insane") and never measured.
# Let's measure it: how long does one pipelined ZADD per follower actually take
# on this machine, and what does that project to at celebrity follower counts?

BENCH_PREFIX = "bench:fanout"

def measure_fanout(n_followers: int, repeats: int = 3) -> float:
    """Median seconds to push ONE post into n_followers Redis feeds, pipelined."""
    r = get_redis()
    samples = []
    for run in range(repeats):
        keys = [f"{BENCH_PREFIX}:{run}:{i}" for i in range(n_followers)]
        start = time.time()
        pipe = r.pipeline()
        for key in keys:
            pipe.zadd(key, {"1": 1.0})
        pipe.execute()
        samples.append(time.time() - start)
        r.delete(*keys)          # clean up after ourselves
    return statistics.median(samples)

print("Fan-out cost vs follower count (pipelined ZADDs against this Redis):\n")
print(f"{'Followers':>12} {'Fan-out time':>14} {'µs / follower':>15}")
print("-" * 44)
timings = {}
for n in (100, 1_000, 10_000):
    seconds = measure_fanout(n)
    timings[n] = seconds
    print(f"{n:>12,} {seconds*1000:>13.1f}ms {seconds/n*1e6:>14.2f}")

# Use the largest sample for the unit cost — small samples are dominated by
# the single round-trip, not by the per-follower work.
per_follower_s = timings[10_000] / 10_000

# Fan-out cost must grow WITH the follower count. If it did not, there would be
# no celebrity problem and nothing below would be worth teaching.
assert timings[10_000] > timings[1_000] * 2, (
    f"fan-out cost is not scaling with follower count: "
    f"1k={timings[1_000]*1000:.1f}ms, 10k={timings[10_000]*1000:.1f}ms")

# ── Now project that unit cost onto real celebrity follower counts ──────
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT u.username, u.display_name, u.follower_count,
           COUNT(f.follower_id) AS actual_followers_in_db
    FROM users u
    LEFT JOIN follows f ON f.followee_id = u.id
    WHERE u.is_celebrity = TRUE
    GROUP BY u.id
    ORDER BY u.follower_count DESC
""")
celebs = cur.fetchall()
conn.close()

REGULAR_FOLLOWERS = 10   # a typical user in this lab's seed data
regular_s = REGULAR_FOLLOWERS * per_follower_s

print(f"\n\nAt {per_follower_s*1e6:.2f}µs per follower, one post costs:\n")
print(f"{'Account':<16} {'Followers':>14} {'Demo DB':>9} {'Redis writes':>14} {'Fan-out time':>14}")
print("-" * 72)
print(f"{'(regular user)':<16} {REGULAR_FOLLOWERS:>14,} {'~10':>9} "
      f"{REGULAR_FOLLOWERS:>14,} {regular_s*1000:>13.2f}ms")
for row in celebs:
    projected = row["follower_count"] * per_follower_s
    print(f"{row['username']:<16} {row['follower_count']:>14,} "
          f"{row['actual_followers_in_db']:>9} {row['follower_count']:>14,} "
          f"{projected:>13.1f}s")

worst = max(celebs, key=lambda c: c["follower_count"])
worst_s = worst["follower_count"] * per_follower_s
amplification = worst["follower_count"] / REGULAR_FOLLOWERS

# Ronaldo is the number everyone quotes in interviews — put a measured number on it.
ronaldo_s = 600_000_000 * per_follower_s
print(f"\n{'Cristiano Ronaldo':<16} {600_000_000:>14,} {'n/a':>9} "
      f"{600_000_000:>14,} {ronaldo_s/60:>11.1f}min")

print(f"\n⚠️  {worst['username']} costs {amplification:,.0f}× the write work of a "
      f"regular user for ONE post")
print(f"   — and that is a single-node, no-network, best-case measurement.")
print(f"   Real fan-out crosses a network to a sharded Redis and retries failures.")

# ── Assertions: the gap has to stay enormous or the hybrid design is pointless
assert amplification >= 100_000, (
    f"expected >=100,000x write amplification for a celebrity, got {amplification:,.0f}x")
assert worst_s > regular_s, "celebrity fan-out must cost more than a regular user's"
assert worst_s > 0.1, (
    f"projected celebrity fan-out of {worst_s:.3f}s looks implausibly cheap — "
    f"unit cost measured at {per_follower_s*1e6:.3f}µs/follower")
assert ronaldo_s > 6.0, (
    f"600M followers should project to at least several seconds, got {ronaldo_s:.1f}s")
print(f"\n✅ Measured, not asserted: fan-out on write is O(followers) and the "
      f"constant is small — the follower count is what kills you.")

## 🔀 Strategy 3: Hybrid Approach (What Instagram Uses)

The solution is to **combine both strategies**:

- **Regular users** (< 100K followers): fan-out on write (push to followers' feeds)
- **Celebrities** (≥ 100K followers): fan-out on read (merge at read time)

### Where does 100K come from?

The threshold is not a magic number — it is the point where the write cost of
one post exceeds the read cost of merging that author in at read time.

Using the unit cost we just measured (call it `w` seconds per follower feed
write) and a read-side merge that costs one extra indexed query per celebrity
followed (call it `q`):

```
write cost of one celebrity post  = followers × w
read cost of NOT fanning out      = followers × refreshes_per_day × q
                                    (every follower pays q on every refresh)
```

Fan-out on write wins while `w < refreshes_per_day × q`. Both sides scale with
follower count, so the follower count alone never decides it — what decides it
is **posting rate vs read rate**. An account with 100K followers that posts once
a week is cheap to fan out; one that posts 20 times a day is not. Instagram's
real rule is closer to "fan-out cost per follower per day", and 100K followers is
the round number that usually crosses it. The exact value is a tuning knob,
not a law — the important part is that there *is* a crossover.

When a user opens their feed:
1. Read precomputed feed from Redis (posts from regular users they follow)
2. Query recent posts from celebrities they follow (fan-out on read)
3. Merge both lists, sort by time

```
User opens feed
      │
      ├──► Redis: get precomputed feed    ──┐
      │    (regular users' posts)            │
      │                                      ├──► Merge + Sort ──► Return
      └──► DB: get celebrity posts         ──┘
           (fan-out on read, just a few)
```

In [ ]:
CELEBRITY_THRESHOLD = 100000  # Users with >= 100K followers are "celebrities"

def create_post_hybrid(author_id: int, caption: str):
    """
    Hybrid post creation:
    - Always save to DB
    - Only fan-out to followers if the author is NOT a celebrity
    """
    conn = get_db()
    cur = conn.cursor()

    # Check if this user is a celebrity
    cur.execute("SELECT follower_count FROM users WHERE id = %s", (author_id,))
    follower_count = cur.fetchone()[0]
    is_celebrity = follower_count >= CELEBRITY_THRESHOLD

    # Save the post
    cur.execute(
        """INSERT INTO posts (author_id, caption, media_type, media_key, created_at)
           VALUES (%s, %s, 'photo', %s, NOW())
           RETURNING id, extract(epoch from created_at)""",
        (author_id, caption, f"photos/user_{author_id}/hybrid_post.jpg")
    )
    post_id, post_ts = cur.fetchone()
    post_ts = float(post_ts)  # Decimal → float for Redis
    conn.commit()
    conn.close()

    if is_celebrity:
        print(f"👑 Celebrity post #{post_id} — skipping fan-out (too many followers)")
        print(f"   {follower_count:,} followers would be too expensive to push to")
    else:
        print(f"👤 Regular user post #{post_id} — fanning out to followers")
        fan_out_on_write(author_id, post_id, post_ts)

    return post_id, is_celebrity

# Regular user posts → fan-out happens
print("=" * 50)
print("Regular user (User 3) posts:")
print("=" * 50)
regular_post_id, regular_is_celeb = create_post_hybrid(
    author_id=3, caption="Regular user post!")

print()

# Celebrity posts → no fan-out
print("=" * 50)
print("Celebrity (celeb_bob, user 52) posts:")
print("=" * 50)
celeb_post_id, celeb_is_celeb = create_post_hybrid(
    author_id=52, caption="Celebrity announcement!")

# ── The branch has to actually branch ───────────────────────────────────
assert not regular_is_celeb and celeb_is_celeb, (regular_is_celeb, celeb_is_celeb)

r = get_redis()
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT follower_id FROM follows WHERE followee_id = 3 ORDER BY follower_id LIMIT 1")
follower_of_regular = cur.fetchone()[0]
cur.execute("SELECT follower_id FROM follows WHERE followee_id = 52 ORDER BY follower_id LIMIT 1")
follower_of_celeb = cur.fetchone()[0]
conn.close()

assert r.zscore(f"feed:{follower_of_regular}", str(regular_post_id)) is not None, (
    f"regular-user post #{regular_post_id} should have been pushed into "
    f"feed:{follower_of_regular}")
assert r.zscore(f"feed:{follower_of_celeb}", str(celeb_post_id)) is None, (
    f"celebrity post #{celeb_post_id} must NOT be fanned out into "
    f"feed:{follower_of_celeb} — skipping that write is the entire point")
print(f"\n✅ Post #{regular_post_id} pushed to feed:{follower_of_regular}; "
      f"post #{celeb_post_id} deliberately absent from feed:{follower_of_celeb}")

In [ ]:
HYBRID_STATS = {}

def get_feed_hybrid(user_id: int, limit: int = 20) -> list:
    """
    Hybrid feed retrieval:
    1. Get precomputed feed from Redis (regular users' posts)
    2. Get recent posts from celebrities we follow (fan-out on read)
    3. Merge and sort
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # ── Part 1: Precomputed feed from Redis ──────────────────
    feed_key = f"feed:{user_id}"
    precomputed_ids = r.zrevrange(feed_key, 0, limit - 1)
    redis_time = (time.time() - start) * 1000

    # ── Part 2: Celebrity posts (fan-out on read) ────────────
    cur.execute("""
        SELECT p.id, p.author_id, p.caption, p.like_count, p.created_at
        FROM posts p
        JOIN follows f ON f.followee_id = p.author_id
        JOIN users u ON u.id = p.author_id
        WHERE f.follower_id = %s
          AND u.follower_count >= %s
        ORDER BY p.created_at DESC
        LIMIT %s
    """, (user_id, CELEBRITY_THRESHOLD, limit))
    celebrity_posts = cur.fetchall()
    celeb_time = (time.time() - start) * 1000 - redis_time

    # ── Part 3: Hydrate precomputed post IDs ─────────────────
    precomputed_posts = []
    if precomputed_ids:
        cur.execute("""
            SELECT id, author_id, caption, like_count, created_at
            FROM posts WHERE id = ANY(%s)
        """, ([int(pid) for pid in precomputed_ids],))
        precomputed_posts = cur.fetchall()

    # ── Part 4: Merge and sort ───────────────────────────────
    # Deduplicate (a celebrity post might appear in both lists)
    seen_ids = set()
    merged = []
    for post in list(precomputed_posts) + list(celebrity_posts):
        if post["id"] not in seen_ids:
            seen_ids.add(post["id"])
            merged.append(post)

    merged.sort(key=lambda p: p["created_at"], reverse=True)
    feed = merged[:limit]
    total_time = (time.time() - start) * 1000

    conn.close()

    HYBRID_STATS.update(precomputed=len(precomputed_ids),
                        celebrity=len(celebrity_posts),
                        returned=len(feed))

    print(f"📊 Hybrid feed stats:")
    print(f"   Redis (precomputed): {redis_time:.1f}ms → {len(precomputed_ids)} post IDs")
    print(f"   DB (celebrity read): {celeb_time:.1f}ms → {len(celebrity_posts)} posts")
    print(f"   Total: {total_time:.1f}ms → {len(feed)} posts returned")

    return feed

# Get hybrid feed for User 1
print("Loading User 1's feed (hybrid approach)...\n")
hybrid_feed = get_feed_hybrid(user_id=1, limit=10)
if hybrid_feed:
    print(f"\nTop posts in hybrid feed:")
    for i, post in enumerate(hybrid_feed, 1):
        print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}")

# ── The read-side half has to actually produce something ────────────────
conn = get_db()
cur = conn.cursor()
cur.execute("""SELECT u.id FROM users u JOIN follows f ON f.followee_id = u.id
               WHERE f.follower_id = 1 AND u.follower_count >= %s""",
            (CELEBRITY_THRESHOLD,))
celebs_followed = {row[0] for row in cur.fetchall()}
conn.close()

assert celebs_followed, "User 1 should follow at least one celebrity in the seed data"
assert HYBRID_STATS["celebrity"] > 0, (
    f"the fan-out-on-read half returned nothing even though user 1 follows "
    f"{sorted(celebs_followed)}")
authors = {post["author_id"] for post in hybrid_feed}
assert authors & celebs_followed, (
    f"no celebrity post survived the merge: authors {sorted(authors)}")
assert len({post["id"] for post in hybrid_feed}) == len(hybrid_feed), (
    "merging the two sources must deduplicate — a post in both lists appeared twice")

print(f"\n✅ Read path contributed {HYBRID_STATS['celebrity']} celebrity posts.")
print(f"   The write path contributed {HYBRID_STATS['precomputed']} — it will be 0")
print(f"   until we load the precomputed feed into Redis in the next section.")

## 📊 Populating Redis Feeds from Existing Data

Our database already has precomputed feed entries (from `init.sql`).  
Let's load them into Redis so the hybrid read path works fully.

In [ ]:
def populate_redis_feeds():
    """
    Load precomputed feeds from PostgreSQL into Redis sorted sets.
    In production, Redis would be the primary feed store.
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    # Get all precomputed feed entries for non-celebrity authors
    cur.execute("""
        SELECT pf.user_id, pf.post_id, extract(epoch from pf.post_created_at) as ts
        FROM precomputed_feed pf
        JOIN users u ON u.id = pf.post_author_id
        WHERE u.follower_count < %s
    """, (CELEBRITY_THRESHOLD,))

    rows = cur.fetchall()
    pipeline = r.pipeline()
    for user_id, post_id, ts in rows:
        pipeline.zadd(f"feed:{user_id}", {str(post_id): float(ts)})
    pipeline.execute()

    conn.close()
    print(f"✅ Loaded {len(rows)} feed entries into Redis")

    # Show some stats
    sample_users = [1, 5, 10, 25]
    for uid in sample_users:
        count = r.zcard(f"feed:{uid}")
        print(f"   feed:{uid} has {count} posts")

populate_redis_feeds()

# Everything below reads feed:1, so fail loudly here rather than three cells later.
assert get_redis().zcard("feed:1") >= 10, (
    f"feed:1 only has {get_redis().zcard('feed:1')} entries — "
    "the seed data or init.sql did not load correctly")

# Now try the hybrid feed again
print("\n" + "=" * 50)
print("Hybrid feed for User 1 (with Redis populated):")
print("=" * 50 + "\n")
hybrid_feed = get_feed_hybrid(user_id=1, limit=10)
if hybrid_feed:
    print(f"\nTop posts:")
    for i, post in enumerate(hybrid_feed, 1):
        print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}")

# With Redis populated, a hybrid that quietly degrades into one strategy is a bug.
assert HYBRID_STATS["precomputed"] > 0, (
    "the fan-out-on-write half is still empty after populate_redis_feeds()")
assert HYBRID_STATS["celebrity"] > 0, "the fan-out-on-read half returned nothing"
print(f"\n✅ Both halves live: {HYBRID_STATS['precomputed']} precomputed IDs from "
      f"Redis + {HYBRID_STATS['celebrity']} celebrity posts from PostgreSQL")
print(f"   Note the celebrity posts dominate the top 10 here purely because the")
print(f"   seed gives them the most recent timestamps — the merge is chronological,")
print(f"   not weighted. A real feed ranks instead of sorting by time.")

## 📄 Paginating the Feed: Offset vs Cursor

Everything so far returns the *first* page. The feed only works if the second
page works too — and this is where a lot of real implementations quietly break.

The obvious approach is **offset pagination**: page 1 is ranks `0..19`,
page 2 is ranks `20..39`. It is correct right up until someone the user follows
posts *between* the two requests. Fan-out pushes that post to rank 0, every
other rank shifts down by one, and the item that was last on page 1 is now the
first item on page 2.

```
t0   page 1 = ranks 0..4    [P9 P8 P7 P6 P5]
t1   new post P10 arrives → fan-out puts it at rank 0
t2   page 2 = ranks 5..9    [P5 P4 P3 P2 P1]
                             ^^ P5 was already on page 1 — the user sees it twice
```

A **delete** does the mirror-image damage: ranks shift *up*, and an item is
skipped entirely — the user never sees it and never knows. Both failures are
silent. No error, no exception, just a wrong feed.

The fix is **cursor (keyset) pagination**. Instead of "skip 5 rows", say
"give me what sorts strictly after *this exact item*". The cursor is the sort
key of the last item returned, so inserts and deletes above the cursor cannot
move the reader's place.

Let's reproduce the duplicate first, then fix it.

In [ ]:
PAGE_SIZE = 5
DEMO_USER = 1

r = get_redis()
feed_key = f"feed:{DEMO_USER}"
assert r.zcard(feed_key) >= 2 * PAGE_SIZE, (
    f"the pagination demo needs at least {2 * PAGE_SIZE} entries in {feed_key}, "
    f"got {r.zcard(feed_key)} — run populate_redis_feeds() above first")


def get_feed_page_offset(user_id: int, page: int, page_size: int = PAGE_SIZE) -> list:
    """❌ Offset pagination: page N is ranks [N*size, N*size + size - 1]."""
    start = page * page_size
    return get_redis().zrevrange(f"feed:{user_id}", start, start + page_size - 1)


def deliver_new_post(author_id: int, follower_id: int) -> int:
    """
    Simulate the fan-out worker delivering ONE brand-new post into one follower's
    feed while that follower is mid-scroll. (fan_out_on_write() would push it to
    every follower; here we only need the one feed we are paginating.)
    """
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        """INSERT INTO posts (author_id, caption, media_type, media_key, created_at)
           VALUES (%s, 'Posted while you were scrolling 📜', 'photo', %s, NOW())
           RETURNING id, extract(epoch from created_at)""",
        (author_id, f"photos/user_{author_id}/mid_scroll.jpg")
    )
    post_id, ts = cur.fetchone()
    conn.commit()
    conn.close()
    get_redis().zadd(f"feed:{follower_id}", {str(post_id): float(ts)})
    return post_id


page1 = get_feed_page_offset(DEMO_USER, page=0)
print(f"Page 1 (offset, ranks 0-4): {page1}")

# User 1 follows user 11 — one of their posts lands between the two requests.
interloper = deliver_new_post(author_id=11, follower_id=DEMO_USER)
print(f"→ Post #{interloper} fanned into feed:{DEMO_USER} (now rank 0)\n")

page2 = get_feed_page_offset(DEMO_USER, page=1)
print(f"Page 2 (offset, ranks 5-9): {page2}")

dupes = sorted(set(page1) & set(page2))
print(f"\n❌ Shown on both pages: {dupes}")
print(f"   Post #{page1[-1]} was rank 4, the insert pushed it to rank 5, "
      f"and rank 5 is where page 2 starts.")

# The bug IS the lesson. If offset pagination ever stops duplicating here, this
# cell has stopped demonstrating anything and should fail loudly.
assert dupes, "offset pagination should have duplicated the page-1 boundary item"
assert page1[-1] in page2, (
    f"expected the last item of page 1 (#{page1[-1]}) to reappear on page 2, got {page2}")

In [ ]:
def get_feed_page_cursor(user_id: int, cursor: tuple | None = None,
                         page_size: int = PAGE_SIZE) -> tuple:
    """
    ✅ Cursor (keyset) pagination over the same Redis sorted set.

    The cursor is the (score, post_id) of the last item we returned.

    The score alone is NOT a safe cursor. Timestamps are not unique, and two
    posts sharing a score would make one of them permanently invisible — the
    exact "skip" bug we are trying to fix, just rarer and harder to notice.

    The tie-break must match how the store itself orders ties, or the cursor and
    the scan disagree at the boundary. Redis orders a sorted set by score, then
    by member **lexicographically** (reversed for ZREVRANGE) — so we compare
    post ids as strings here, not as ints. `"9"` sorts after `"10"` in Redis,
    and this comparison has to agree.

    Returns (page_of_post_ids, next_cursor).
    """
    r = get_redis()
    key = f"feed:{user_id}"

    if cursor is None:
        rows = r.zrevrange(key, 0, page_size - 1, withscores=True)
    else:
        last_score, last_id = cursor
        # Anchor on the score INCLUSIVELY so ties are not skipped, then drop
        # everything at the boundary score we already returned. The overfetch
        # covers the ties; a production version would keep paging while the
        # whole batch still sits on the boundary score.
        raw = r.zrevrangebyscore(
            key, last_score, "-inf", start=0, num=page_size + 20, withscores=True
        )
        rows = [(pid, s) for pid, s in raw
                if (s, pid) < (last_score, last_id)][:page_size]

    page = [pid for pid, _ in rows]
    next_cursor = (rows[-1][1], rows[-1][0]) if rows else None
    return page, next_cursor


# Replay the exact same interleaving: read page 1, someone posts, read page 2.
c_page1, cursor = get_feed_page_cursor(DEMO_USER)
print(f"Page 1 (cursor): {c_page1}")
print(f"   next cursor = (score={cursor[0]:.3f}, post_id={cursor[1]!r})")

interloper2 = deliver_new_post(author_id=11, follower_id=DEMO_USER)
print(f"→ Post #{interloper2} fanned into feed:{DEMO_USER} (now rank 0)\n")

c_page2, cursor = get_feed_page_cursor(DEMO_USER, cursor)
print(f"Page 2 (cursor): {c_page2}")

overlap = sorted(set(c_page1) & set(c_page2))
print(f"\n✅ Shown on both pages: {overlap or 'nothing'}")

assert not overlap, f"cursor pagination must not duplicate items, got {overlap}"
assert len(c_page2) == PAGE_SIZE, f"page 2 should be full, got {len(c_page2)}"
combined = c_page1 + c_page2
assert len(set(combined)) == len(combined), "cursor pages must be disjoint"
assert str(interloper2) not in combined, (
    f"post #{interloper2} arrived above the cursor and must not appear "
    f"half-way down the scroll")

print(f"\n💡 Post #{interloper2} is in neither page. It sorts *above* the cursor,")
print(f"   so it shows up on the next pull-to-refresh — not spliced into the")
print(f"   middle of the user's scroll, and not at the cost of a duplicate.")
print(f"\n   Cost note: offset pagination on a ZSET is O(offset + n) because")
print(f"   Redis walks the skip list to reach the offset. Cursor pagination is")
print(f"   O(log N + n) — it seeks straight to the score. Deep pages are where")
print(f"   the offset version gets slow as well as wrong.")

## 🧹 Invalidating the Precomputed Feed

Fan-out on write **copies** data. Every copy is a cache, and every cache can go
stale. The most common way an Instagram-style feed goes wrong in production is
an **unfollow**: the follow edge is deleted in PostgreSQL, but the posts already
pushed into that follower's Redis feed are still sitting there. The user
unfollows someone and keeps seeing their posts for days.

The same hole exists for a **deleted post**, a **blocked account**, and a post
that is still `pending` upload — nothing removes them from the sorted set.

Two honest fixes, and real systems use both:

| Approach | Cost | Covers |
|----------|------|--------|
| **Invalidate on write** — `ZREM` that author's posts from the follower's feed | O(posts by author) per unfollow | Unfollow, block, post delete |
| **Filter on read** — treat the ZSET as a *superset*, drop rows during hydration | One predicate in the hydration query | Everything, including missed invalidations |

We already added the read-side filter (`media_upload_status = 'complete'`).
Let's show why the write-side one is also needed.

In [ ]:
def posts_by_author_in_feed(user_id: int, author_id: int) -> list:
    """Which of `author_id`'s posts are still sitting in this Redis feed?"""
    r = get_redis()
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT id FROM posts WHERE author_id = %s", (author_id,))
    author_posts = {str(row[0]) for row in cur.fetchall()}
    conn.close()
    return sorted(pid for pid in r.zrange(f"feed:{user_id}", 0, -1)
                  if pid in author_posts)


def unfollow(follower_id: int, followee_id: int, invalidate: bool):
    """Delete the follow edge — and, if we remember to, the feed entries it produced."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute("DELETE FROM follows WHERE follower_id = %s AND followee_id = %s",
                (follower_id, followee_id))
    cur.execute("DELETE FROM precomputed_feed WHERE user_id = %s AND post_author_id = %s",
                (follower_id, followee_id))
    cur.execute("SELECT id FROM posts WHERE author_id = %s", (followee_id,))
    post_ids = [str(row[0]) for row in cur.fetchall()]
    conn.commit()
    conn.close()

    if invalidate and post_ids:
        get_redis().zrem(f"feed:{follower_id}", *post_ids)
    return post_ids


STALE_AUTHOR = 25            # User 1 follows user 25 in the seed data
before = posts_by_author_in_feed(DEMO_USER, STALE_AUTHOR)
assert before, (
    f"expected user {STALE_AUTHOR}'s posts in feed:{DEMO_USER} — "
    "run populate_redis_feeds() above first")
print(f"feed:{DEMO_USER} holds {len(before)} posts by user {STALE_AUTHOR}: {before}")

# ❌ Unfollow WITHOUT invalidation — PostgreSQL is right, Redis is not.
unfollow(DEMO_USER, STALE_AUTHOR, invalidate=False)
stale = posts_by_author_in_feed(DEMO_USER, STALE_AUTHOR)
print(f"\n❌ After unfollow (no invalidation): {len(stale)} of their posts "
      f"are STILL in the feed")
print(f"   The follow row is gone. Nothing tells Redis about that.")
assert stale == before, (
    "the unfollow should have left the Redis feed completely untouched — "
    "that is the bug this cell exists to show")

# ✅ Run the invalidation the unfollow should have done.
removed = get_redis().zrem(f"feed:{DEMO_USER}", *stale)
after = posts_by_author_in_feed(DEMO_USER, STALE_AUTHOR)
print(f"\n✅ After ZREM: {removed} entries removed, {len(after)} left in the feed")
assert after == [], f"invalidation must clear the unfollowed author's posts, {after} left"

# ── Restore the follow edge and its feed entries so this notebook re-runs ──
conn = get_db()
cur = conn.cursor()
cur.execute("""INSERT INTO follows (follower_id, followee_id) VALUES (%s, %s)
               ON CONFLICT DO NOTHING""", (DEMO_USER, STALE_AUTHOR))
cur.execute("""INSERT INTO precomputed_feed (user_id, post_id, post_author_id, post_created_at)
               SELECT %s, p.id, p.author_id, p.created_at
               FROM posts p WHERE p.author_id = %s
               ON CONFLICT DO NOTHING""", (DEMO_USER, STALE_AUTHOR))
cur.execute("SELECT id, extract(epoch from created_at) FROM posts WHERE author_id = %s",
            (STALE_AUTHOR,))
restore_rows = cur.fetchall()
conn.commit()
conn.close()
get_redis().zadd(f"feed:{DEMO_USER}",
                 {str(pid): float(ts) for pid, ts in restore_rows})
assert posts_by_author_in_feed(DEMO_USER, STALE_AUTHOR), "restore failed"
print(f"\n(restored follow {DEMO_USER} → {STALE_AUTHOR} so the notebook is re-runnable)")
print(f"\n💡 Invalidation cost is O(posts by that author), which is fine for an")
print(f"   unfollow but ruinous for a celebrity deleting a post — 600M ZREMs.")
print(f"   That is another reason celebrity posts never enter the ZSET at all.")

## 🧠 Key Takeaways

1. **Fan-out on Read** — simple but slow at scale (N+1 queries per feed load)
2. **Fan-out on Write** — fast reads (single Redis lookup) but expensive writes for popular users
3. **The Celebrity Problem** — a user with 600M followers can't fan-out on write
4. **Hybrid approach** — fan-out on write for regular users, fan-out on read for celebrities
5. **Redis sorted sets** — perfect data structure for precomputed feeds (ZADD, ZREVRANGE)
6. **Cursor pagination** — offset pagination duplicates items whenever a post is
   inserted between two page requests, and skips items when one is deleted.
   Page by "what sorts after this item", never by "skip N rows"
7. **Invalidation** — a precomputed feed is a cache. Unfollows, blocks and post
   deletes must remove entries on the write side, and the read side needs a
   filter so a missed invalidation degrades into a short page, not a wrong one

### Interview Tips

- Start with fan-out on read (show you understand the simple approach)
- Explain why it doesn't scale (do the math: 500M users × 500 follows × 5 refreshes/day)
- Propose fan-out on write as the improvement
- Identify the celebrity problem before the interviewer asks
- Land on the hybrid approach — this is what Instagram actually uses
- Volunteer the pagination question before the interviewer does: "how do you page
  a feed that is being written to while the user scrolls?"
- Say out loud that the celebrity threshold is a tuning knob driven by posting
  rate vs read rate, not a fixed 100K

### What This Toy Version Does NOT Do

- **No async fan-out.** `fan_out_on_write` runs inline; production queues it
  (Kafka → worker pool) so the poster's request returns immediately
- **No ranking.** This feed is strictly chronological. Instagram's is ranked,
  which changes the pagination story (the sort key is a score that can move)
- **No sharding.** One Redis holds every feed; at 500M users the feeds are
  sharded by user id, and fan-out becomes a scatter across shards
- **No retries or idempotency.** A crashed fan-out worker loses those pushes;
  production tracks a high-water mark per follower so it can resume

### What's Next?

In **Notebook 3**, we'll explore **Stories** — Instagram's ephemeral content  
that disappears after 24 hours. We'll use Redis TTL to handle automatic expiration.